# 02 · Pit Stop Strategy Optimisation

Uses the degradation models from `01` to find the fastest strategy.

**Model:** race time = Σ stint tyre losses + stops × pit loss (fuel and SC laps cancel between strategies).
Brute-forces every legal 1- and 2-stop plan (two dry compounds required), then stress-tests the best ones with a Monte Carlo safety-car simulation.

In [ ]:
# --- Setup (works locally or on Colab) ---
import sys, os
if "google.colab" in sys.modules and not os.path.exists("f1strat"):
    REPO = "https://github.com/anerudh-k/f1-tyre-strategy"
    !git clone -q {REPO} repo && cp -r repo/* .
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from f1strat import *
COLORS = {"SOFT": "#e10600", "MEDIUM": "#f5c400", "HARD": "#8a8a8a"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
os.makedirs("figures", exist_ok=True); os.makedirs("data", exist_ok=True)

In [ ]:
# --- Config ---
MODEL_FILE = "data/deg_2025_bahrain.json"   # from notebook 01 ("data/deg_demo.json" for demo)
# or from notebook 03 (generic model), e.g. "data/deg_generic_2026_sao_paulo.json"
PIT_LOSS = None      # None = use estimate from data; or set a known value, e.g. 22.0
PIT_LOSS_SC = 11.0   # stop under safety car
P_SC = 0.5           # probability of at least one safety car in the race

if not os.path.exists(MODEL_FILE) and "generic" in MODEL_FILE:
    raise FileNotFoundError(f"{MODEL_FILE} not found - run notebook 03 first")
if not os.path.exists(MODEL_FILE):   # e.g. fresh Colab runtime: rebuild models from notebook 01's defaults
    print("Model file not found - fitting it now (same steps as notebook 01)...")
    _, _year, _gp = MODEL_FILE.replace(".json", "").split("_", 2)
    _, _laps = load_race(int(_year), _gp.replace("_", " ").title())
    _tl = int(_laps["LapNumber"].max())
    _m = fit_all(fuel_correct(clean_laps(_laps), _tl))
    os.makedirs("data", exist_ok=True)
    save_models(_m, MODEL_FILE, meta=dict(year=int(_year), gp=_gp.replace("_", " ").title(),
                total_laps=_tl, pit_loss_est=pit_loss_from_session(_laps)))
models, meta = load_models(MODEL_FILE)
TOTAL_LAPS = meta["total_laps"]
est = meta.get("pit_loss_est")
PIT_LOSS = PIT_LOSS or (est if est and est == est else 21.0)
if meta.get("p_sc") is not None and meta["p_sc"] == meta["p_sc"]:
    P_SC = meta["p_sc"]   # circuit's historical safety-car rate (generic model)
setup = RaceSetup(TOTAL_LAPS, pit_loss=PIT_LOSS, pit_loss_sc=PIT_LOSS_SC, models=models)
print(meta.get("event"), meta.get("year"), "| laps:", TOTAL_LAPS, "| pit loss:", round(PIT_LOSS, 1),
      "s | P(SC):", round(P_SC, 2))
print({c: setup.max_stint(c) for c in models}, "<- max stint per compound")

## 1. Deterministic optimum (no safety car)

In [ ]:
strats = enumerate_strategies(setup)
best = best_per_sequence(strats)
print(f"{len(strats):,} legal strategies evaluated")
best[["Stops", "Compounds", "PitLaps", "Stints", "Gap"]].head(10).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
one = strats[strats["Stops"] == 1]
for comp, g in one.groupby("Compounds"):
    g = g.sort_values("PitLaps")
    ax.plot([p[0] for p in g["PitLaps"]], g["Gap"], lw=2, label=comp)
two_best = strats[strats["Stops"] == 2]["Gap"].min()
ax.axhline(two_best, c="k", ls="--", lw=1, label="best 2-stop")
ax.set(xlabel="Pit lap", ylabel="Gap to optimum (s)", title="One-stop: race time vs pit lap",
       ylim=(-1, max(15, two_best + 5)))
ax.legend(); plt.savefig("figures/one_stop_window.png", bbox_inches="tight")

In [ ]:
# Two-stop map for the best two-stop compound sequence
seq = strats[strats["Stops"] == 2].iloc[0]["Seq"]
g = strats[strats["Seq"] == seq]
grid = np.full((TOTAL_LAPS, TOTAL_LAPS), np.nan)
for p, t in zip(g["PitLaps"], g["Gap"]):
    grid[p[1], p[0]] = t
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(grid, origin="lower", cmap="viridis_r", vmax=np.nanmin(grid) + 10)
ax.set(xlabel="Stop 1 lap", ylabel="Stop 2 lap", title=f"Two-stop {'-'.join(seq)}: gap to optimum (s)")
plt.colorbar(im); plt.savefig("figures/two_stop_map.png", bbox_inches="tight")

## 2. Sensitivity: when does a two-stop win?

In [ ]:
rows = []
for pl in np.arange(16, 28.5, 1.0):
    s = RaceSetup(TOTAL_LAPS, pit_loss=pl, models=models)
    st = enumerate_strategies(s)
    b1 = st[st.Stops == 1].iloc[0]; b2 = st[st.Stops == 2].iloc[0]
    rows.append(dict(pit_loss=pl, best_1stop=f"{b1.Compounds} {b1.PitLaps}",
                     best_2stop=f"{b2.Compounds} {b2.PitLaps}", two_minus_one=b2.RaceTime - b1.RaceTime))
sens = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(sens.pit_loss, sens.two_minus_one, "o-"); ax.axhline(0, c="k", lw=1)
ax.set(xlabel="Pit loss (s)", ylabel="2-stop minus 1-stop (s)", title="Below 0 → two-stop is faster")
plt.savefig("figures/pit_loss_sensitivity.png", bbox_inches="tight"); sens.round(2)

## 3. Monte Carlo: robustness to safety cars
Each simulated race may get a safety car at a random lap. If a planned stop is within 6 laps, the car pits under SC instead (cheaper stop). Win rate = how often each plan is the fastest.

In [ ]:
cands = best.head(6)
mc, times = monte_carlo(setup, cands, n_sims=5000, p_sc=P_SC)
mc.sort_values("MeanTime").round(2)

In [ ]:
order = mc["MeanTime"].argsort().to_numpy()
labels = [f"{mc.Compounds[i]} {mc.PitLaps[i]}" for i in order]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].boxplot([times[:, i] - times.min() for i in order], vert=False, whis=(5, 95), showfliers=False)
ax[0].set_yticklabels(labels); ax[0].set(xlabel="Relative race time (s)", title="Race time distribution")
ax[1].barh(labels, mc["WinRate"].to_numpy()[order]); ax[1].invert_yaxis()
ax[0].invert_yaxis(); ax[1].set(xlabel="Win rate", title="How often each plan is fastest")
plt.tight_layout(); plt.savefig("figures/monte_carlo.png", bbox_inches="tight")

## 4. Compare with what teams actually did

In [ ]:
try:
    _, laps_raw = load_race(meta["year"], meta["gp"]) if not MODEL_FILE.endswith("demo.json") else (None, None)
except Exception as e:   # e.g. predicting a race that hasn't happened yet
    print("No actual race data to compare with:", e); laps_raw = None
if laps_raw is not None:
    res = (laps_raw.groupby(["Driver", "Stint"]).agg(Compound=("Compound", "first"),
           Laps=("LapNumber", "count")).reset_index())
    actual = res.groupby("Driver").apply(lambda g: " → ".join(f"{c[0]}{n}" for c, n in zip(g.Compound, g.Laps)))
    final = laps_raw.groupby("Driver")["Position"].last().sort_values()
    display(pd.DataFrame({"Finish": final, "Strategy": actual}).head(10))

## Findings
*(Your own words after running on real data.)*
- Optimal strategy and pit window vs what the top teams ran.
- Pit-loss threshold where a two-stop becomes faster.
- How a safety car changes the picture (win rates).
- Limitations: no track position/overtaking difficulty, no undercut/overcut modelling, single SC per race, deg fitted from one race.

**Next step:** add an undercut model (out-lap on new tyres vs in-lap on old) and track-position penalty.